In [1]:
import pandas as pd
import numpy as np
import joblib
import snowflake.connector

In [2]:
conn = snowflake.connector.connect(

    user="KUMAR3006",

    password="@Jyokumar_143@",

    account="xvcevgk-xi43656",

    warehouse="SALES_WH",

    database="SALES_FORECAST_DB",

    schema="SALES_SCHEMA"

)
cursor = conn.cursor()
print(cursor)
print("connected successfully")

connected successfully


In [5]:
model = joblib.load(
   "C:/Users/kumar/Downloads/pre_val.pkl"
)

In [7]:
print(type(model))

<class 'sklearn.ensemble._forest.RandomForestRegressor'>


In [9]:
df = pd.read_csv(
   "C:/Users/kumar/Downloads/final_ml_data.csv"
)

In [11]:
df.head()

,PRODUCT_ID,SALE_DATE,SALES_REP,REGION,SALES_AMOUNT,QUANTITY_SOLD,PRODUCT_CATEGORY,UNIT_COST,UNIT_PRICE,CUSTOMER_TYPE,...,MONTH,PROFIT,YEAR,DAY,QUARTER,WEEK,DAY_OF_WEEK,PROFIT_MARGIN,DISCOUNT_AMOUNT,REVENUE_PER_UNIT
0,1052,2023-02-03,Bob,1,5053.97,18,3,152.75,267.22,1,...,2,2060.46,2023,3,1,5,4,40.769138,0.240498,280.776111
1,1093,2023-04-21,Bob,3,4384.02,17,3,3816.39,4209.44,1,...,4,6681.85,2023,21,2,16,4,152.413766,4.630384,257.883529
2,1015,2023-09-21,David,2,4631.23,30,2,261.56,371.40,1,...,9,3295.20,2023,21,3,38,3,71.151724,0.742800,154.374333
3,1072,2023-08-24,Bob,2,2167.94,39,0,4330.03,4467.75,0,...,8,5371.08,2023,24,3,34,3,247.750399,0.893550,55.588205
4,1061,2023-03-24,Charlie,0,3750.20,13,1,637.37,692.71,0,...,3,719.42,2023,24,1,12,4,19.183510,0.554168,288.476923


In [13]:
X = df.drop(
    columns=[
        "SALE_DATE",
        "SALES_AMOUNT",
        "SALES_REP","PAYMENT_METHOD","REGION_AND_SALES_REP","SALES_REP"
    ],
    errors="ignore"
)

In [15]:
print(X.columns)

Index(['PRODUCT_ID', 'REGION', 'QUANTITY_SOLD', 'PRODUCT_CATEGORY',
       'UNIT_COST', 'UNIT_PRICE', 'CUSTOMER_TYPE', 'DISCOUNT', 'SALES_CHANNEL',
       'MONTH', 'PROFIT', 'YEAR', 'DAY', 'QUARTER', 'WEEK', 'DAY_OF_WEEK',
       'PROFIT_MARGIN', 'DISCOUNT_AMOUNT', 'REVENUE_PER_UNIT'],
      dtype='object')


In [17]:
predictions = model.predict(X)

In [19]:
df["PREDICTED_SALES"] = predictions

In [21]:
df[["SALES_AMOUNT", "PREDICTED_SALES"]].head()

,SALES_AMOUNT,PREDICTED_SALES
0,5053.97,5026.9249
1,4384.02,4576.6147
2,4631.23,4601.0029
3,2167.94,2141.2741
4,3750.20,3750.0263


In [33]:
df["MODEL_NAME"] = "Random Forest"

In [29]:
prediction_df = df[
    [
        "SALE_DATE",
        "REGION",
        "PRODUCT_CATEGORY",
        "SALES_AMOUNT",
        "PREDICTED_SALES",
        "MODEL_NAME"
    ]
]
prediction_df.head()

KeyError: "['MODEL_NAME'] not in index"

In [49]:
print(prediction_df.shape)

(1000, 6)


In [3]:
for row in prediction_df.itertuples(index=False):
    try:
        cursor.execute(
            """
            INSERT INTO SALES_PREDICTIONS
            (
                SALE_DATE,
                REGION,
                PRODUCT_CATEGORY,
                ACTUAL_SALES,
                PREDICTED_SALES,
                MODEL_NAME
            )
            VALUES (%s, %s, %s, %s, %s, %s)
            """,
            (
                row.SALE_DATE,
                row.REGION,
                row.PRODUCT_CATEGORY,
                row.SALES_AMOUNT,
                row.PREDICTED_SALES,
                row.MODEL_NAME
            )
        )
    except Exception as e:
        print("Error inserting row:", row)
        print(e)

conn.commit()

NameError: name 'prediction_df' is not defined

In [41]:
all_predictions = model.predict(X)

prediction_df = df.copy()
prediction_df["PREDICTED_SALES"] = all_predictions
prediction_df["MODEL_NAME"] = "Random Forest"

print(len(prediction_df))

1000
